# 01 · Data Collection & Preprocessing 🧹  ·  *Role A*

**Your mission:** turn messy raw satellite tiles into a tidy table of little building patches:
`(pre_patch, post_patch, damage_label, disaster_type)`, split **by disaster type**.

When you're done, everyone else can start. You're the foundation of the whole project. 💪

**Where to get data:** xView2 / xBD — https://xview2.org/ (request access, grab train + tier3).  
Alternatives if needed: Maxar Open Data, Copernicus EMS, Sentinel-2 (lower resolution).

### 📁 What the raw data looks like
After you download & unzip xBD, you should have:
```
data/raw/xbd/
  images/  <event>_<id>_pre_disaster.png   <event>_<id>_post_disaster.png
  labels/  <event>_<id>_pre_disaster.json  <event>_<id>_post_disaster.json
```
Each **post**-disaster JSON lists building shapes, each tagged with a damage level:
`no-damage | minor-damage | major-damage | destroyed | un-classified`.
(Tip: ignore `un-classified` — it's not a real label.)

In [ ]:
# First, let's just check whether the data is downloaded. No data yet? That's fine —
# you can read through the whole notebook and fill in the TODOs before downloading.
import sys, os; sys.path.append(os.path.abspath('..'))
from src import config, data_utils
import json, glob
import numpy as np
from pathlib import Path

RAW = Path(config.DATA_DIR) / 'raw' / 'xbd'
print('Looking for raw data in:', RAW.resolve())
print('Found it?', RAW.exists())

### Step 1 — Read one label file 📖
Each post-disaster JSON has a list of buildings. For each one we want its **shape** (a polygon
in `wkt` text format) and its **damage level**. The disaster name is hiding in the filename.

👉 *TODO(Role A):* the function below is a working starting point — try it on a real file.

In [ ]:
def parse_label(json_path):
    """Read a post-disaster JSON -> list of buildings + the disaster name."""
    with open(json_path) as f:
        data = json.load(f)
    buildings = []
    for feat in data['features'].get('xy', []):          # 'xy' = pixel coordinates
        subtype = feat['properties'].get('subtype', 'un-classified')
        buildings.append({'wkt': feat['wkt'], 'subtype': subtype})
    # filename starts with the event, e.g. 'hurricane-michael_00000_post_disaster'
    disaster = Path(json_path).stem.split('-')[0]          # -> 'hurricane'
    return buildings, disaster

# Try it once data exists (uncomment):
# sample = next(iter(glob.glob(str(RAW/'labels'/'*post*.json'))), None)
# buildings, disaster = parse_label(sample)
# print(disaster, '->', len(buildings), 'buildings; first:', buildings[0])

### Step 2 — Cut a patch around each building ✂️
Good news: xBD pre & post tiles are **already lined up** (same grid), so co-registration is
mostly free here. For each building polygon we find its bounding box, add a little padding,
and crop the **same window** from both the pre and post image. Then resize to a fixed size.

👉 *TODO(Role A):* handle buildings near the tile edge, skip `un-classified`, maybe skip tiny ones.

In [ ]:
# Helper bits. (pip install shapely  if shapely is missing.)
from shapely import wkt as shapely_wkt
from PIL import Image

def building_bbox(wkt_str):
    """Polygon text -> (minx, miny, maxx, maxy) pixel box."""
    minx, miny, maxx, maxy = shapely_wkt.loads(wkt_str).bounds
    return int(minx), int(miny), int(maxx), int(maxy)

def crop_patch(img, bbox, size=config.PATCH_SIZE, pad=10):
    """Crop a padded box from one image and resize to a square patch."""
    minx, miny, maxx, maxy = bbox
    return img.crop((minx-pad, miny-pad, maxx+pad, maxy+pad)).resize((size, size))

### Step 3 — Throw away bad patches 🌥️
Clouds, smoke and corrupted tiles will confuse the model. A simple, good-enough filter:
drop patches that are almost all white (cloud) or almost all black (missing data).

👉 *TODO(Role A):* tune the thresholds; add more checks if you see junk in the EDA later.

In [ ]:
def is_bad_patch(patch_np, white_frac=0.5):
    """True if the patch looks like cloud/blank and should be skipped."""
    gray = patch_np.mean(axis=-1)
    too_white = (gray > 240).mean() > white_frac
    too_dark  = (gray < 15).mean()  > white_frac
    return too_white or too_dark

### Step 4 — Build the patch table + the disaster split 🗂️
Loop over every tile, save each `(pre, post)` patch pair to disk, and record a row in a CSV.

⭐ **The most important rule of the whole project:** the test set is a disaster type the model
*never trained on*. Train/val on `config.TRAIN_DISASTERS`, test only on `config.HELDOUT_DISASTER`.
Please don't use a random split — graders specifically check for this.

In [ ]:
import pandas as pd

PROC = Path(config.DATA_DIR) / 'processed'
(PROC / 'patches').mkdir(parents=True, exist_ok=True)

def assign_split(disaster_type):
    """Held-out disaster -> 'test'. Everything else -> 'train' (carve 'val' out later)."""
    if disaster_type == config.HELDOUT_DISASTER:
        return 'test'
    return 'train'

# Sketch of the main loop (fill in):
# rows = []
# for label_json in glob.glob(str(RAW/'labels'/'*post*.json')):
#     buildings, disaster = parse_label(label_json)
#     ... crop pre & post patches, skip bad ones, save PNGs ...
#     rows.append({'pre_path':..., 'post_path':..., 'label':config.NAME_TO_IDX[subtype],
#                  'disaster_type':disaster, 'split':assign_split(disaster)})
# manifest = pd.DataFrame(rows)
# manifest.to_csv(PROC / 'manifest.csv', index=False)
# print(manifest['split'].value_counts())

### Step 5 — Deal with class imbalance ⚖️
Reality: ~90% of buildings are `no-damage`. If we do nothing, the model 'wins' by always
guessing no-damage and learns nothing useful. Easiest fix: give rare classes a bigger weight
in the loss. (Alternatives: oversample rare classes, or focal loss — your call, just justify it.)

👉 *TODO(Role A):* compute the weights from the **train** split and save them for C & D.

In [ ]:
# counts  = manifest[manifest.split=='train']['label'].value_counts().sort_index()
# weights = counts.sum() / (len(counts) * counts)   # rarer class -> bigger weight
# np.save(PROC / 'class_weights.npy', weights.values)
# print('class counts:\n', counts)
# print('class weights:\n', weights)

### 🎁 You're done when these two files exist:
- `data/processed/manifest.csv`
- `data/processed/class_weights.npy`

Drop a message in the group chat — B, C and D can start the second these land. Nice work! 🙌